In [64]:
import pickle
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import pandas as pd

from src_dataloader.data_generator_pytorch import DataGenerator, OrganizeData

In [21]:
TRAIN_DATA_PATH = '/data/dev/ml_skyportal/AJs_Stuff/alt_data_train'
TEST_DATA_PATH = '/data/dev/ml_skyportal/AJs_Stuff/alt_data_test'

In [56]:
train_df = pd.read_csv('train_df.csv')
test_df = pd.read_csv('test_df.csv')

data_test, data_train = OrganizeData.create_df_of_object_alerts_in_dataset(test_df, train_df, TEST_DATA_PATH, TRAIN_DATA_PATH, 'type_step1')

In [59]:
# leave 10 the most common classes
CLASSES = ['SN Ia', 'SN II', 'SN IIP', 'Cataclysmic', 'AGN', 'SN IIn', 'SN Ic', 'SN Ib', 'SN IIb', 'Tidal Disruption Event']
data_train = data_train[data_train['type'].isin(CLASSES)]

In [60]:
# downsample SN Ia
sn_ia = data_train[data_train['type'] == 'SN Ia'].sample(n=4922, random_state=42)
data_train = pd.concat([data_train[data_train['type'] != 'SN Ia'], sn_ia])

In [61]:
train_files, val_files  = OrganizeData.split_and_compute_class_weights(data_train, 'type_step1', verbose=True)

In [74]:
train_dataset = DataGenerator(TRAIN_DATA_PATH, data_train, step='type_step1', file_list=train_files)
val_dataset = DataGenerator(TRAIN_DATA_PATH, data_train, step='type_step1', file_list=val_files)

In [75]:
photometry, metadata, images, spectra, label = train_dataset[0]

In [76]:
photometry.shape, metadata.shape, images.shape, spectra.shape, label

(torch.Size([180, 4]),
 torch.Size([10]),
 torch.Size([63, 63, 3]),
 torch.Size([214, 2]),
 tensor(6, dtype=torch.int8))

In [72]:
def plot_dataset_item(dataset, item_id):
    """
    Plots photometry, images, and spectra for a specific dataset item in a single row.

    Parameters:
    dataset: The dataset object containing data items.
    item_id (int): The ID of the dataset item to plot.
    """
    # Extract data for the given ID
    photometry, metadata, images, spectra, labels = dataset[item_id]

    fig = plt.figure(figsize=(24, 6))
    # Define GridSpec with 6 columns for a single row layout
    gs = GridSpec(1, 5, width_ratios=[1, 1, 1, 1.5, 1.5])

    # Image plots in the first three columns
    ax_img1 = fig.add_subplot(gs[0, 0])
    ax_img1.imshow(images[:, :, 0], cmap='gray')
    ax_img1.axis('off')
    ax_img1.set_title('Image Channel 1')

    ax_img2 = fig.add_subplot(gs[0, 1])
    ax_img2.imshow(images[:, :, 1], cmap='gray')
    ax_img2.axis('off')
    ax_img2.set_title('Image Channel 2')

    ax_img3 = fig.add_subplot(gs[0, 2])
    ax_img3.imshow(images[:, :, 2], cmap='gray')
    ax_img3.axis('off')
    ax_img3.set_title('Image Channel 3')

    # Photometry plot in the fourth column
    ax_photometry = fig.add_subplot(gs[0, 3])
    ax_photometry.plot(photometry[:, 0], photometry[:, 1], label='Photometry 1')
    ax_photometry.plot(photometry[:, 0], photometry[:, 2], label='Photometry 2')
    ax_photometry.plot(photometry[:, 0], photometry[:, 3], label='Photometry 3')
    ax_photometry.set_title('Photometry')
    ax_photometry.legend()

    # Spectra plot in the last two columns
    ax_spectra = fig.add_subplot(gs[0, 4])
    ax_spectra.plot(spectra[:, 0], spectra[:, 1], label='Spectra')
    ax_spectra.set_title('Spectra')
    ax_spectra.legend()

    # Adjust layout
    plt.tight_layout()
    plt.show()

In [71]:
labels

tensor([1.1349e-29, 0.0000e+00, 0.0000e+00, 0.0000e+00, 2.2421e-44, 0.0000e+00,
        1.1210e-44, 0.0000e+00])